## MiddleWare
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

* Tracking agent behavior with logging, analytics, and debugging.
* Transforming prompts, tool selection, and output formatting.
* Adding retries, fallbacks, and early termination logic.
* Applying rate limits, guardrails, and PII detection.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

###### NOTE:
The checkpointer=InMemorySaver() argument enables state persistence for the agent.

What it does:

Saves the agent's state (messages, variables, context) to memory after each invocation
Allows the agent to retrieve previous state when you use the same thread_id
Without it, each call to agent.invoke() would start fresh with no conversation history

In this example:
checkpointer=InMemorySaver()

InMemorySaver() stores state in memory (not disk). When you call:
response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)

The checkpointer saves the response to the thread "test-1". On the next call with the same thread_id, it loads that saved state, allowing the agent to reference previous messages.

Other options:

InMemorySaver() — keeps state in RAM (lost when process ends)
SqliteSaver() — persists to disk
Custom implementations for databases, files, etc.
Without a checkpointer, the thread_id wouldn't work — you'd have no way to retrieve past conversation state.

#### Summarization middleware
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

##### Messages size Summarization

In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [21]:
# Run with a thread ID
config = {"configurable": {"thread_id": "test-1"}}

In [22]:
questions = [
    "what is 2+2?",
    "what is 3+3?",
    "what is 10*5?",
    "what is 100/4?",
    "what is 2^10?",
    "what is the square root of 144?",
    "what is the factorial of 5?",
]

for q in questions:
    response = agent.invoke(
        {"messages":[HumanMessage(content=q)]}, 
        config
    )
    print(f"Messages: {response}")
    print(f"Response: {response['messages'][-1].content}")
    print(f"Messages length: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='1568f1f5-c78f-4752-b05b-2023181d6c26'), AIMessage(content='2 + 2 equals 4.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 14, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_60952bc00e', 'id': 'chatcmpl-E5slj41t1Un60x3VCTDKIpSGlDjbS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f9e84-0727-7322-9a68-e41298c49b8d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 8, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': 

##### Token size Summarization

In [26]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotel(city: str) -> str:
    """Search hotels in specific city, returns long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model="gpt-4o-mini",
        trigger=("tokens", 550),
        keep=("tokens", 200)
    )]
)

config = {"configurable": {"thread_id": "test-1"}}

#Token counter (approximate)
def count_tokens(messages) -> int:
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars ≈ 1 token


In [27]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~143 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='edf7b9b4-a33e-4cc8-bebf-8700a5042d39'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 56, 'total_tokens': 71, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_4e2e0e5f15', 'id': 'chatcmpl-E5tBxU0ZPRwUnYRtQbmvwhObziEEk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f9e9c-da05-7ce2-b338-8032f618ad5d-0', tool_calls=[{'name': 'search_hotel', 'args': {'city': 'Paris'}, 'id': 'call_nASoR15J3LlaL3313d0NQidv', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 56,